# RAG Support Assistant remote benchmark

Run this notebook inside Google Colab. It keeps the local Windows machine and the iMac as thin clients only: no local Ollama, no local GraceKelly, no Docker, and no local model pulls.

The safe default cells run a mock provider benchmark first. The live direct-Mistral benchmark is opt-in inside the notebook and reads `MISTRAL_API_KEY` with `getpass`, so the key is not printed.

In [ ]:
# Runtime probe: this runs in Colab, not on the local/iMac machines.
import os, platform, subprocess

print(platform.platform())
print(platform.python_version())
print("CPU count:", os.cpu_count())
print("Memory:")
print("\n".join(open('/proc/meminfo', encoding='utf-8').read().splitlines()[:3]))
print("GPU:")
gpu = subprocess.run(['bash', '-lc', 'command -v nvidia-smi >/dev/null && nvidia-smi -L || true'], text=True, capture_output=True)
print(gpu.stdout.strip() or 'No GPU visible')

## Clone the repository

If the local branch contains commits that are not pushed, push them first or upload an archive manually. Colab can only clone what GitHub can see. The default branch below is the current Colab runbook branch; switch it to `master` after this work lands there.

In [ ]:
REPO_URL = 'https://github.com/brownjuly2003-code/RAG_Support_Assistant.git'
BRANCH = 'colab-remote-benchmark'

!rm -rf /content/RAG_Support_Assistant
!git clone --depth 1 --branch "$BRANCH" "$REPO_URL" /content/RAG_Support_Assistant
%cd /content/RAG_Support_Assistant
!git rev-parse --short HEAD
!git status --short --branch

In [ ]:
# Install project dependencies inside Colab. This may take several minutes.
# It does not install or start Ollama/GraceKelly/Docker.
%pip install -q -r requirements.txt

In [ ]:
# Safe smoke: mock provider benchmark, no external provider calls and no DB persistence.
!python scripts/regression_eval.py \
  --baseline ministral-3b-latest \
  --candidate mistral-small-latest \
  --dataset evaluation/curated_cases.jsonl \
  --tenant all \
  --max-cases 3 \
  --seed 42 \
  --no-persist

## Optional live direct-Mistral benchmark

This cell spends API quota but does not use local model RAM. Keep `--max-cases` small for the first run. Do not use `ollama-small`, `gk-fast`, `gk-strong`, or `gracekelly-mixed` here; those imply local services or browser-backed orchestration.

In [ ]:
from getpass import getpass
import os

RUN_LIVE = False  # Change to True only when you intentionally want paid API calls from Colab.

if RUN_LIVE:
    if not os.environ.get('MISTRAL_API_KEY'):
        os.environ['MISTRAL_API_KEY'] = getpass('MISTRAL_API_KEY: ')
    !python scripts/regression_eval.py \
      --baseline ministral-3b-latest \
      --candidate mistral-small-latest \
      --dataset evaluation/curated_cases.jsonl \
      --tenant all \
      --max-cases 3 \
      --seed 42 \
      --allow-paid-apis \
      --no-persist
else:
    print('RUN_LIVE is False; live paid API benchmark was skipped.')

In [ ]:
# List generated reports for download from the Colab file browser.
!find reports/regression -maxdepth 1 -type f -printf '%TY-%Tm-%Td %TH:%TM %p\n' 2>/dev/null | sort | tail -20 || true